## 1. Obtain the data

The data it will be marketing_kpis_looker.csv, it is obtained in ETL.

In [13]:
import pandas as pd
import numpy as np

DATA_FOLDER = "../Marketing_Predict/data/"
FILE_PATH = DATA_FOLDER + "marketing_kpis_looker.csv"

df = pd.read_csv(FILE_PATH)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   campaign_id          200000 non-null  int64  
 1   company              200000 non-null  object 
 2   campaign_type        200000 non-null  object 
 3   target_audience      200000 non-null  object 
 4   duration             200000 non-null  object 
 5   channel_used         200000 non-null  object 
 6   conversion_rate      200000 non-null  float64
 7   acquisition_cost     200000 non-null  float64
 8   roi                  200000 non-null  float64
 9   location             200000 non-null  object 
 10  language             200000 non-null  object 
 11  clicks               200000 non-null  int64  
 12  impressions          200000 non-null  int64  
 13  engagement_score     200000 non-null  int64  
 14  customer_segment     200000 non-null  object 
 15  kpi_ctr_pct      

## 2. Features and target

* Features → the variables the model will use to learn
* Target → what the model has to predict (ROI)

I only use the columns that represent decisions that I can control when launching a campaign, which channel to use, what type of campaign, to which audience, in which city, in which language, and for how long.

In [14]:
FEATURES = [
    'campaign_type',     
    'channel_used',      
    'customer_segment',  
    'target_audience',   
    'location',          
    'language',          
    'duration'          
]

TARGET = 'roi'

## 3. Cleaning data

In [15]:
df['duration'] = (df['duration']
                  .astype(str)
                  .str.replace(' days', '', regex=False)
                  .str.replace(' day', '', regex=False)
                  .str.strip()
                  )

df['duration'] = pd.to_numeric(df['duration'], errors ='coerce').astype(int)

df.head(5)

,campaign_id,company,campaign_type,target_audience,duration,channel_used,conversion_rate,acquisition_cost,roi,location,...,engagement_score,customer_segment,kpi_ctr_pct,dim_year,dim_month,dim_month_name,dim_quarter,dim_year_month,segment_roi,segment_acquisition
0,1,Innovate Industries,Email,Men 18-24,30,Google Ads,0.04,16174.0,6.29,Chicago,...,6,Health & Wellness,26.3267,2021,1,January,Q1,2021-01,High (>6),Very Expensive (>15k)
1,2,NexGen Systems,Email,Women 35-44,60,Google Ads,0.12,11566.0,5.61,New York,...,7,Fashionistas,1.5419,2021,1,January,Q1,2021-01,Medium (3-6),Expensive (10k-15k)
2,3,Alpha Innovations,Influencer,Men 25-34,30,YouTube,0.07,10200.0,7.18,Los Angeles,...,1,Outdoor Adventurers,7.5864,2021,1,January,Q1,2021-01,High (>6),Expensive (10k-15k)
3,4,DataTech Solutions,Display,All Ages,60,YouTube,0.11,12724.0,5.55,Miami,...,7,Health & Wellness,11.9231,2021,1,January,Q1,2021-01,Medium (3-6),Expensive (10k-15k)
4,5,NexGen Systems,Email,Men 25-34,15,YouTube,0.05,16452.0,6.50,Los Angeles,...,3,Health & Wellness,9.0217,2021,1,January,Q1,2021-01,High (>6),Very Expensive (>15k)


## 4. Encoding of categorical variables

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

df_model = df[FEATURES + [TARGET]].copy()
df_model.dropna(inplace=True)

encoders = {}
categorical_cols = ['campaign_type', 'channel_used', 'customer_segment',
                    'target_audience', 'location', 'language']

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le
    print(f"{col}: {list(le.classes_)}")

campaign_type: ['Display', 'Email', 'Influencer', 'Search', 'Social Media']
channel_used: ['Email', 'Facebook', 'Google Ads', 'Instagram', 'Website', 'YouTube']
customer_segment: ['Fashionistas', 'Foodies', 'Health & Wellness', 'Outdoor Adventurers', 'Tech Enthusiasts']
target_audience: ['All Ages', 'Men 18-24', 'Men 25-34', 'Women 25-34', 'Women 35-44']
location: ['Chicago', 'Houston', 'Los Angeles', 'Miami', 'New York']
language: ['English', 'French', 'German', 'Mandarin', 'Spanish']


## 5. Training the model

In [17]:
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [18]:
model = RandomForestRegressor(
    n_estimators=200,   
    max_depth=10,       
    random_state=42,    
    n_jobs=-1           
)

model.fit(X_train, y_train)

# Evaluate the model
print("Evaluate the model on the test set...")

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n" + "="*50)
print("Evaluation metrics")
print("="*50)
print(f"   MAE  (Mean Absolute Error):        {mae:.4f}")
print(f"   RMSE (Root Mean Square Error):      {rmse:.4f}")
print(f"   R²   (R-squared):     {r2:.4f}")
print("="*50)

if r2 >= 0.7:
    print("   The model has good predictive power")
elif r2 >= 0.5:
    print("   The model has moderate predictive power")
else:
    print("   The model has low predictive power")
    print("      This may be due to the dataset being synthetic")
    print("      and the ROI was generated randomly without real patterns")

Evaluate the model on the test set...

Evaluation metrics
   MAE  (Mean Absolute Error):        1.5049
   RMSE (Root Mean Square Error):      1.7390
   R²   (R-squared):     -0.0041
   The model has low predictive power
      This may be due to the dataset being synthetic
      and the ROI was generated randomly without real patterns


## 6. Conclusions

A Random Forest Regressor model was trained to predict ROI based on campaign variables. The model achieved an R² of -0.004, indicating no predictive pattern between the input variables and ROI. This finding is consistent with the synthetic nature of the dataset, where ROI values were generated randomly without a real relationship to campaign type, channel or audience segment. This result demonstrates the importance of data quality in machine learning a model is only as good as the patterns present in the data.